<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07 -->
<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'yes do both (OpenAI testing cleanup before the Task 3 submission run)', Date: 2026-10-07 -->

# Task 3: agentic equity research

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sathudeva7/CDAZZDEV-MLE-SATHURSAN/blob/main/task3_agentic/task3_agentic_research.ipynb)

This notebook is a thin driver. The logic lives in the `task3_agentic/` modules,
which reuse Task 1's data, indicators and news code, and carry offline tests in
`tests/`. Each section shows the evidence for one or more rubric rows. Results are
printed by code cells only, so no number in the text can drift from the run.

| Rubric row (marks) | Section | Where the logic lives |
|---|---|---|
| 3A All five tools implemented (15) | 3A.1 | `task3_agentic/tools.py`, `schemas.py` |
| 3A Autonomous tool selection (10) | 3A.2, 3A.5 (the order table) | `task3_agentic/agent.py` |
| 3A Observe and replan cycle (8) | 3A.2 (live trace), 3A.3 | `agent.py`, `printer.py` |
| 3A Final report quality (10) | 3A.4 | `task3_agentic/report.py` |
| 3A Error handling (7) | 3A.5 | `tools.py` (envelope + hint), `agent.py` |
| 3B Distinct roles and tool restriction (8) | 3B.3 | `pipeline.py`, `handoff.py` |
| 3B Structured handoff schema (8) | 3B.2 | `schemas.py` (`DataBrief`, `ClarificationRequest`, `ClarificationResponse`) |
| 3B Message trace visible (6) | 3B.1 | `printer.py` |
| 3B Critique loop (8) | 3B.1, 3B.2 | `pipeline.py`, `handoff.py` |
| 3B End-to-end automation (5) | 3B.1 | `pipeline.py` |
| 3C Short-term memory (5) | 3C.1 | `agent.py` (`InMemorySaver`, one thread per conversation) |
| 3C Persistent cache (5) | 3C.2 | `task3_agentic/cache.py` |
| 3C agent_trace.jsonl (5) | 3C.3 | `task3_agentic/trace.py` |

The query, from the brief: *Analyse the current financial health and market
sentiment of AAPL. Identify the top three risks to its share price over the next
90 days and suggest one data-driven hedge strategy.*

**Models.** The submitted run uses the `free` LLM profile: Groq's free tier
(`openai/gpt-oss-120b`), with an OpenRouter free model as fallback. The setup
cell prints the profile and model. The agents' tool-calling model and every
structured answer (sentiment labels, the brief, the clarification request, the
reports) come from that profile. Budgets per agent run: 8 tool calls and 6
model turns.

**To run it yourself:** add the keys the setup cell lists under Colab's Secrets
(the key icon), then Runtime → Run all.

## Setup

In [1]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'yes do both (OpenAI testing cleanup before the Task 3 submission run)', Date: 2026-10-07
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED = time.perf_counter()
TICKER = "AAPL"
BAD_TICKER = "NOTATICKERZZ"  # a symbol that does not exist, for the error-handling demos
REPO_URL = "https://github.com/sathudeva7/CDAZZDEV-MLE-SATHURSAN"
REPO_REF = "main"  # the branch or tag the Colab run clones
# Set here, not from the shell, so a local LLM_PROFILE cannot leak in.
NOTEBOOK_PROFILE = "free"  # submitted runs are free tier (CLAUDE.md rule 5)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content") / REPO_URL.rsplit("/", 1)[-1]
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    # Locally the notebook runs from task3_agentic/, one level below the repo root.
    ROOT = Path.cwd().parent if Path.cwd().name == "task3_agentic" else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))
os.environ["LLM_PROFILE"] = NOTEBOOK_PROFILE

from common.llm import enable_console_logging
from common.llm_config import active_profile, secret

enable_console_logging()
profile = active_profile()
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()  # the hash is informational
print(f"Running in {'Colab' if IN_COLAB else 'a local kernel'}, repo at {commit or 'unknown commit'}")
print(f"LLM profile: {profile.name} (primary {profile.primary.name}: {profile.primary.model}; "
      f"fallback {profile.fallback.name + ': ' + profile.fallback.model if profile.fallback else 'none'})")
for provider in (profile.primary, profile.fallback):  # presence only, never the value
    if provider is not None:
        print(f"  {'✓' if secret(provider.api_key_env) else '✗'} {provider.api_key_env}")

Running in Colab, repo at 7b69f36
LLM profile: free (primary groq: openai/gpt-oss-120b; fallback openrouter: nvidia/nemotron-3-super-120b-a12b:free)
  ✓ GROQ_API_KEY
  ✗ OPENROUTER_API_KEY


In [2]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# One run, one record: the trace, the call log and today's cache file are cleared,
# so the committed files match this notebook's outputs and the first 3B run is a real cache miss.
import json
from datetime import datetime

import pandas as pd
from IPython.display import Markdown, display

from task1_financial.pipeline import MARKET_TIMEZONE
from task3_agentic.cache import CACHE_DIR, cache_path
from task3_agentic.trace import DEFAULT_LOG_DIR, TRACE_FILE_NAME, read_trace

TODAY = datetime.now(MARKET_TIMEZONE).date()  # the market's date in New York, as every tool uses
TRACE = DEFAULT_LOG_DIR / TRACE_FILE_NAME
CALL_LOG = DEFAULT_LOG_DIR / profile.log_file
CACHE_FILE = cache_path(CACHE_DIR, TICKER, TODAY)
for path in (TRACE, CALL_LOG, CACHE_FILE):
    path.unlink(missing_ok=True)
    print("cleared", path.relative_to(ROOT))

RUN_LABELS: dict[str, str] = {}  # run_id -> section, for the trace summary in 3C.3


def tool_call_count() -> int:
    """tool_call lines written to agent_trace.jsonl so far, by every run."""
    return sum(event["event"] == "tool_call" for event in read_trace(TRACE))

cleared task3_agentic/logs/agent_trace.jsonl
cleared task3_agentic/logs/llm_calls.jsonl
cleared task3_agentic/cache/AAPL_2026-10-07.json


## 3A.1 The five tools, called directly

Each tool returns the same envelope, `ToolResult`: `status` (`ok`, `empty` or
`error`), the typed `data`, `error`, `warnings`, and a `hint` naming an
alternative tool when the status is not ok. Tools never raise. Below, each one is
called by code, with no agent, on AAPL. `llm_sentiment` labels the headlines that
`get_news` returned.

In [3]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.tools import ToolSession

direct = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[direct.run_id] = "3A.1 direct tools"
results = {
    "get_price_data": direct.get_price_data(TICKER, "6mo"),
    "calculate_volatility": direct.calculate_volatility(TICKER, 30),
    "get_news": direct.get_news(TICKER, 10),
}
headlines = [h.title for h in results["get_news"].data.headlines] if results["get_news"].data else []
results["llm_sentiment"] = direct.llm_sentiment(headlines)
results["web_search"] = direct.web_search(f"{TICKER} stock analyst outlook next quarter")


def key_fact(result) -> str:
    """One line from each tool's typed data, read from its fields."""
    d = result.data
    if d is None:
        return result.error or "no data"
    if result.tool == "get_price_data":
        change = f"{d.period_return_pct:+.1f}%" if d.period_return_pct is not None else "n/a"
        return f"{len(d.bars)} bars to {d.as_of}, close {d.bars[-1].close:.2f}, 6-month return {change}"
    if result.tool == "calculate_volatility":
        return f"{d.window}-day vol {d.current_pct:.1f}% (1y median {d.median_1y_pct:.1f}%, percentile {d.percentile_1y:.0f})"
    if result.tool == "get_news":
        return f"{len(d.headlines)} headlines, newest: {d.headlines[0].title[:60]}" if d.headlines else "no headlines"
    if result.tool == "llm_sentiment":
        score = f"{d.summary.score:+.2f}" if d.summary.score is not None else "n/a"
        return f"Sentiment score {score} ({d.summary.label}), {d.summary.scored} scored"
    return f"{len(d.hits)} hits via the {d.backend} backend; first: {d.hits[0].title[:50]}" if d.hits else "no hits"


durations = {e["tool"]: e["duration_ms"] for e in read_trace(TRACE, run_id=direct.run_id) if e["event"] == "tool_call"}
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(
    [
        {"tool": name, "status": r.status, "data type": type(r.data).__name__, "key fact": key_fact(r), "duration_ms": durations.get(name)}
        for name, r in results.items()
    ]
)

INFO:task1_financial.data:AAPL: 501 daily bars from 2024-10-08 to 2026-10-07
INFO:task3_agentic.tools:price cache hit for AAPL
INFO:task1_financial.data:retrying AAPL yahoo news in 1s (attempt 2 of 3): HTTPError: HTTP Error 429: Too Many Requests
INFO:task1_financial.data:retrying AAPL yahoo news in 2s (attempt 3 of 3): HTTPError: HTTP Error 429: Too Many Requests
INFO:task1_financial.news:AAPL yahoo feed: 0 items, 0 recent, 0 new
INFO:task1_financial.news:AAPL google feed: 100 items, 53 recent, 52 new


,tool,status,data type,key fact,duration_ms
0,get_price_data,ok,PriceData,"128 bars to 2026-10-07, close 336.89, 6-month return +33.1%",881.0
1,calculate_volatility,ok,VolatilityStats,"30-day vol 21.1% (1y median 23.1%, percentile 33)",4.7
2,get_news,ok,NewsList,"10 headlines, newest: Morgan Stanley Is Bullish on Apple—But With a Catch",4135.1
3,llm_sentiment,ok,SentimentStats,"Sentiment score -0.01 (neutral), 10 scored",5035.7
4,web_search,ok,SearchResults,5 hits via the text backend; first: Apple (AAPL) Stock Forecast & Analyst Price Target,2216.7


One full envelope, as JSON. The agent never reads this whole object: it reads a
compact **digest** (`ToolResult.for_llm()`), shown second, which keeps a price
history of 120+ bars down to the latest values, period statistics and the last 5 bars.

In [4]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print(results["calculate_volatility"].model_dump_json(indent=2))
price = results["get_price_data"]
print(f"\nget_price_data: the typed result holds {len(price.data.bars)} bars; the digest the agent reads is {len(price.for_llm()):,} characters:")
print(price.for_llm()[:900], "...")

{
  "tool": "calculate_volatility",
  "status": "ok",
  "data": {
    "ticker": "AAPL",
    "window": 30,
    "as_of": "2026-10-07",
    "current_pct": 21.128713932921094,
    "min_1y_pct": 10.286551499034633,
    "median_1y_pct": 23.084780359806928,
    "max_1y_pct": 41.150307688507525,
    "percentile_1y": 32.53968253968254,
    "observations": 252
  },
  "error": null,
  "warnings": [],
  "hint": null
}

get_price_data: the typed result holds 128 bars; the digest the agent reads is 893 characters:
{"tool": "get_price_data", "status": "ok", "data": {"ticker": "AAPL", "period": "6mo", "as_of": "2026-10-07", "close": 336.89, "period_return_pct": 33.13, "period_high": 345.34, "period_low": 245.26, "high_52w": 345.34, "low_52w": 242.76, "close_vs_sma_50_pct": 4.54, "close_vs_sma_200_pct": 16.27, "indicators": {"sma_50": 322.24, "sma_200": 289.73, "rsi_14": 57.9, "macd": 3.3, "macd_signal": 4.11, "macd_hist": -0.81, "bb_upper": 341.74, "bb_middle": 334.49, "bb_lower": 327.24, "bb_pct_b": 

The argument schema each tool exposes to the model. Every tool also takes a
required `why`: the model's one-sentence reason for the call, which the trace
records. It is required because both models tested left the message text empty
when they called tools, so `why` is the only place their reasoning shows.

In [5]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.tools import AGENT_TOOL_ARGS, TOOL_DESCRIPTIONS, TOOL_NAMES

for name in TOOL_NAMES:
    properties = AGENT_TOOL_ARGS[name].model_json_schema()["properties"]
    args = ", ".join(f"{arg}: {spec.get('type', 'enum' if 'enum' in spec else '?')}" for arg, spec in properties.items())
    print(f"{name}({args})")
    print(f"    {TOOL_DESCRIPTIONS[name][:150]}")

get_price_data(why: string, ticker: string, period: string)
    Daily adjusted prices for a ticker with technical indicators (SMA-50, SMA-200, RSI-14, MACD, Bollinger Bands and %B, 30-day volatility). Returns the l
calculate_volatility(why: string, ticker: string, window: integer)
    Annualised historical volatility of daily log returns over `window` trading days, and where today's value sits within the past year (min, median, max,
get_news(why: string, ticker: string, n: integer)
    Recent news headlines (last 7 days) for a ticker, newest first, from Yahoo Finance and Google News.
llm_sentiment(why: string, headlines: array)
    Labels each headline positive, negative or neutral for the share price using an LLM, and returns the Sentiment score from -1 (uniformly negative) to +
web_search(why: string, query: string)
    Searches the web (DuckDuckGo) and returns the top 5 results with snippets. Use it for analyst commentary, price targets, upcoming events and risks tha


## 3A.2 The research agent

One agent with all five tools, built as a small LangGraph `StateGraph`:
`agent → tools → agent → … → finish`. The model decides which tools to call and
in what order. Nothing in the code fixes a sequence: the graph only loops until
the model stops calling tools or the budget (8 calls, 6 turns) runs out. Then the
`finish` step writes the report.

The trace below streams as the run goes. `[single] turn n` is a model turn, `->`
a tool it chose, with its `why`, and `ok` / `x` what came back.

In [6]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.agent import MAX_TOOL_CALLS, ResearchAgent
from task3_agentic.printer import print_replan_cycles, print_update

session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[session.run_id] = "3A.2 research agent"
agent = ResearchAgent(session)
run = agent.research(TICKER, on_update=print_update)
main_order = [obs["tool"] for obs in run.observations]
print(f"\n{run.tool_calls} tool calls (budget {MAX_TOOL_CALLS}), thread {run.thread_id}")
print("order the model chose:", " -> ".join(main_order))

[single] turn 1 (openai/gpt-oss-120b, 1,030 tokens)
  -> get_price_data(period='1mo', ticker='AAPL')
     why: Need recent price trend, momentum indicators, and volatility to assess health and unusual movement


INFO:task1_financial.data:AAPL: 501 daily bars from 2024-10-08 to 2026-10-07


[single] tools
  ok get_price_data: {"ticker": "AAPL", "period": "1mo", "as_of": "2026-10-07", "close": 336.93, "period_return_pct": 6.55, "period_high": 345.34, "period_low": 309.9, "high_52w": 3...


INFO:task3_agentic.tools:price cache hit for AAPL


[single] turn 2 (openai/gpt-oss-120b, 1,499 tokens)
  -> calculate_volatility(ticker='AAPL')
     why: Having price trend, need to know if current volatility is unusually high to assess risk
[single] tools
  ok calculate_volatility: {"ticker": "AAPL", "window": 30, "as_of": "2026-10-07", "current_pct": 21.13, "min_1y_pct": 10.29, "median_1y_pct": 23.08, "max_1y_pct": 41.15, "percentile_1y":...


...

[single] turn 5 (openai/gpt-oss-120b, 2,645 tokens)
  -> web_search(query='Apple AAPL risk factors next 90 days insider selling token trading plan regulatory')
     why: Identify analyst commentary on near-term risks not captured in headlines


INFO:openai._base_client:Retrying request in 23.000000 seconds (retry 1 of 4)


[single] tools
  ok web_search: {"query": "Apple AAPL risk factors next 90 days insider selling token trading plan regulatory", "backend": "text", "hits": [{"title": "Apple (AAPL) Insider Trad...
[single] turn 6 (openai/gpt-oss-120b, 3,664 tokens)
  says: Apple’s share price has risen ≈ 6.5 % over the past month, trading 4.6 % above its 50‑day SMA and 16 % above its 200‑day SMA, with RSI at a modest 57.9 and a modest MACD‑histogram‑negative signal. 30‑day historical volatility is 21 % annualised – roughly the 32nd percentile of the past year, so the move is not unusually volatile.  Sentiment from the ten most recent headlines is essentially neutral (LLM score ≈ ‑0.01), with the strongest positive note being the 9 % earnings‑guidance lift and the strongest negatives coming from large insider sales (Tim Cook and the new CEO together sold > $70 m)


INFO:openai._base_client:Retrying request in 24.000000 seconds (retry 1 of 4)
INFO:openai._base_client:Retrying request in 1.000000 seconds (retry 2 of 4)
INFO:openai._base_client:Retrying request in 1.000000 seconds (retry 3 of 4)


[single] report written by llm: 3 risks, hedge protective_put, 0 warnings

5 tool calls (budget 8), thread 4d87a05b1995
order the model chose: get_price_data -> calculate_volatility -> get_news -> llm_sentiment -> web_search


## 3A.3 Observe → replan

Read back from `agent_trace.jsonl`: each cycle pairs the tool results one model
turn received with the calls the **next** turn chose, and the reason it gave.
Calls made in the same turn were chosen together, so the cycle boundary is the turn.

In [7]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print_replan_cycles(read_trace(TRACE, run_id=session.run_id), agent.name)

cycle into turn 2
  observed get_price_data -> ok: {"tool": "get_price_data", "status": "ok", "data": {"ticker": "AAPL", "period": "1mo", "as_of": "2026-10-07", "close": 336.93, "period_return_pct": 6.55, "perio
  chose    calculate_volatility(ticker='AAPL', window=30)
           why: Having price trend, need to know if current volatility is unusually high to assess risk
cycle into turn 3
  observed calculate_volatility -> ok: {"tool": "calculate_volatility", "status": "ok", "data": {"ticker": "AAPL", "window": 30, "as_of": "2026-10-07", "current_pct": 21.13, "min_1y_pct": 10.29, "med
  chose    get_news(ticker='AAPL', n=10)
           why: Need recent headlines to gauge sentiment and identify emerging risks
cycle into turn 4
  observed get_news -> ok: {"tool": "get_news", "status": "ok", "data": {"ticker": "AAPL", "count": 10, "headlines": [{"title": "Morgan Stanley Is Bullish on Apple\u2014But With a Catch",
  chose    llm_sentiment(headlines=[10 items])
           why: Need sentimen

## 3A.4 The final report

The report is written by one structured LLM call from the tool digests, then
**checked by code**. Evidence citing a tool that did not return a usable result
is dropped, with a warning. The hedge's price levels are computed by code (the
expected 90-day move = price × volatility × √(63/252), the SMA-200, the 52-week
low, the lower Bollinger band). The LLM only picks a strategy and the levels by
name. If the LLM fails, a template report is written from the same data.
`generated_by` says which happened.

In [8]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
report = run.report
print(f"generated_by: {report.generated_by} | tools called: {report.tools_called} | warnings: {report.warnings or 'none'}")
display(Markdown(report.to_markdown()))

generated_by: llm | tools called: ['get_price_data', 'calculate_volatility', 'get_news', 'llm_sentiment', 'web_search'] | warnings: none


# AAPL research report (2026-10-07)

## Financial Health Summary

Apple is trading at $336.93, which is 4.56% above its 50‑day SMA (322.24) and 16.29% above its 200‑day SMA (289.73), indicating a strong uptrend. Momentum is mixed: the 14‑day RSI sits at 57.94 (near neutral) while the MACD histogram is –0.8, a slight bearish divergence. Volatility is relatively calm, with today’s 30‑day volatility percentile at 32.54, well below the 50‑percentile threshold. News sentiment is overall neutral (LLM sentiment score –0.007) with a positive headline scoring 0.85 (“Apple stock heads toward November 2 earnings with 9% guidance”) and two negative insider‑selling headlines scoring –0.82 and –0.80.

## Top Three Risks

1. **Insider Selling Pressure**: Recent large sales by senior insiders could signal reduced confidence and weigh on the share price if investors interpret the sales as a negative outlook.
   - Tim Cook Sells $63.8 Million in Apple Stock After Restricted Shares Vest _(source: get_news)_
   - Apple’s New CEO Just Sold $8.5 Million of Stock _(source: get_news)_
   - sentiment score –0.82 for Tim Cook selling headline _(source: llm_sentiment)_
2. **Token‑Trading Plan Regulatory Uncertainty**: The filing to enter a 24/7 token trading plan may attract regulatory scrutiny, potentially causing a short‑term sell‑off or heightened volatility.
   - Apple (AAPL) Shares Enter 24 7 Token Trading Plan In 63 Stock Filing _(source: get_news)_
   - headline from web search: "Apple (AAPL) Insider Trading: $4.3M Sold in 90 Days by 1 ..." _(source: web_search)_
3. **App Store Growth Slowdown Concerns**: Analyst notes of slowing App Store revenue could dampen earnings expectations, pressuring the stock if growth fails to meet forecasts.
   - UBS reiterates Apple stock neutral on slowing App Store growth _(source: get_news)_
   - Apple stock heads toward November 2 earnings with 9% guidance (positive sentiment score 0.85) _(source: llm_sentiment)_

## Hedge Strategy Recommendation

**Protective Put**

- buy put at 301.33 (one_sd_low)

With volatility at a low 32.54 percentile, protective puts are relatively cheap, making outright downside protection cost‑effective. The put shields against the three identified risks: insider selling, regulatory uncertainty from the token‑trading plan, and potential earnings pressure from slowing App Store growth. Buying the put at the one_sd_low level (≈$301.33) caps loss while preserving upside if the stock remains above that band.

Expected 63-trading-day move: ±35.60 (10.6%) from 336.93, using 21.1% volatility (calculate_volatility, 30-day).

_Option premiums are not priced: no tool returns option chains. Levels are indicative strikes from the expected move and price levels; price the legs from a live chain before acting._

In [9]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# The same report as the validated Pydantic object.
print(report.model_dump_json(indent=2))

{
  "ticker": "AAPL",
  "as_of": "2026-10-07",
  "financial_health_summary": "Apple is trading at $336.93, which is 4.56% above its 50‑day SMA (322.24) and 16.29% above its 200‑day SMA (289.73), indicating a strong uptrend. Momentum is mixed: the 14‑day RSI sits at 57.94 (near neutral) while the MACD histogram is –0.8, a slight bearish divergence. Volatility is relatively calm, with today’s 30‑day volatility percentile at 32.54, well below the 50‑percentile threshold. News sentiment is overall neutral (LLM sentiment score –0.007) with a positive headline scoring 0.85 (“Apple stock heads toward November 2 earnings with 9% guidance”) and two negative insider‑selling headlines scoring –0.82 and –0.80.",
  "top_risks": [
    {
      "title": "Insider Selling Pressure",
      "explanation": "Recent large sales by senior insiders could signal reduced confidence and weigh on the share price if investors interpret the sales as a negative outlook.",
      "evidence": [
        {
          "fact

## 3C.1 Short-term memory: a follow-up from context

The agent keeps each conversation in LangGraph's `InMemorySaver`, under a
`thread_id`. The follow-up goes to the same thread, so the earlier tool results
are still in its messages. Its tools stay bound, so whether it calls one is the
model's choice. The cell counts the trace's `tool_call` lines before and after.

In [10]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
FOLLOW_UP = (
    "What annualised volatility did you find, and where does it sit in its one-year range? "
    "Answer from what you already retrieved."
)
before = tool_call_count()
follow_up = agent.ask(FOLLOW_UP, thread_id=run.thread_id, on_update=print_update)
new_lines = tool_call_count() - before
print(f"\nanswer: {follow_up.answer}")
print(f"\ntool calls in the follow-up: {follow_up.tool_calls}; new tool_call lines in agent_trace.jsonl: {new_lines}")
print("Answered from memory, without re-calling a tool." if new_lines == 0 else "The model re-called a tool for this follow-up (see the trace above).")

INFO:openai._base_client:Retrying request in 31.000000 seconds (retry 1 of 4)


[single] turn 1 (openai/gpt-oss-120b, 3,715 tokens)
  says: The 30‑day annualised historical volatility for AAPL is **21.13 %**. Within the past year this sits at roughly the **32nd percentile** (the low‑mid part of its one‑year volatility range, which spans from about 10 % to 41 %).

answer: The 30‑day annualised historical volatility for AAPL is **21.13 %**. Within the past year this sits at roughly the **32nd percentile** (the low‑mid part of its one‑year volatility range, which spans from about 10 % to 41 %).

tool calls in the follow-up: 0; new tool_call lines in agent_trace.jsonl: 0
Answered from memory, without re-calling a tool.


## 3A.5 Error handling

Three demos, from the tool level up:

1. **Tools on a bad ticker, no agent.** Each tool returns an `error` or `empty`
   envelope with a hint, and nothing raises.
2. **A tool that always fails.** `failing_tools=["get_news"]` is a demo switch,
   off by default, that makes `get_news` return an error. The agent reads the
   error and its hint, and tries another route.
3. **The agent on a bad ticker.** Every data tool fails. The run still ends with a
   report that names the gaps, rather than a traceback.

In [11]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
bad = ToolSession(today=TODAY, subject=BAD_TICKER)
RUN_LABELS[bad.run_id] = "3A.5 bad ticker, direct"
bad_results = [
    bad.get_price_data(BAD_TICKER),
    bad.calculate_volatility(BAD_TICKER),
    bad.get_news(BAD_TICKER),
    bad.llm_sentiment([]),
    bad.web_search(f"{BAD_TICKER} stock"),
]
pd.DataFrame(
    [{"tool": r.tool, "status": r.status, "error / warnings": r.error or "; ".join(r.warnings) or "", "hint": r.hint} for r in bad_results]
)

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKERZZ"}}}
ERROR:yfinance:$NOTATICKERZZ: possibly delisted; no timezone found
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTATICKERZZ']: possibly delisted; no timezone found
INFO:task1_financial.data:retrying NOTATICKERZZ price history in 1s (attempt 2 of 3): empty result
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NOTATICKERZZ"}}}
ERROR:yfinance:$NOTATICKERZZ: possibly delisted; no timezone found
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTATICKERZZ']: possibly delisted; no timezone found
INFO:task1_financial.data:retrying NOTATICKERZZ price history in 2s (attempt 3 of 3): empty result
ERROR:yfinance:$NOTATICKERZZ: possibly delisted; no timezone found
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NOTATICKERZZ']: possibly delisted; no t

,tool,status,error / warnings,hint
0,get_price_data,empty,no NOTATICKERZZ price history after 3 attempts (last problem: empty result); check the ticker symbol or try later,Check that NOTATICKERZZ is a valid Yahoo Finance symbol; calculate_volatility reads the same prices. Alternatives: w...
1,calculate_volatility,empty,no NOTATICKERZZ price history after 3 attempts (last problem: empty result); check the ticker symbol or try later,"A shorter window needs less history. Alternatives: get_price_data: its indicators include hv_30, the 30-day annualis..."
2,get_news,empty,no NOTATICKERZZ yahoo news after 3 attempts (last problem: HTTPError: HTTP Error 429: Too Many Requests); check the ...,Alternatives: web_search: search 'NOTATICKERZZ stock news this week' for recent headlines.
3,llm_sentiment,empty,no headlines were given,Alternatives: web_search: search for analyst commentary on NOTATICKERZZ and judge its tone from the snippets.
4,web_search,ok,,None


In [12]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
failing = ToolSession(today=TODAY, subject=TICKER, failing_tools=["get_news"])
RUN_LABELS[failing.run_id] = "3A.5 get_news failing"
failing_run = ResearchAgent(failing).research(TICKER, on_update=print_update)
failing_order = [obs["tool"] for obs in failing_run.observations]
print()
print_replan_cycles(read_trace(TRACE, run_id=failing.run_id), "single")

[single] turn 1 (openai/gpt-oss-120b, 1,027 tokens)
  -> get_price_data(period='1mo', ticker='AAPL')
     why: Need recent price trend, momentum indicators, and volatility to assess price trend and unusual volatility


INFO:task1_financial.data:AAPL: 501 daily bars from 2024-10-08 to 2026-10-07


[single] tools
  ok get_price_data: {"ticker": "AAPL", "period": "1mo", "as_of": "2026-10-07", "close": 336.91, "period_return_pct": 6.54, "period_high": 345.34, "period_low": 309.9, "high_52w": 3...


INFO:task3_agentic.tools:price cache hit for AAPL
INFO:openai._base_client:Retrying request in 12.000000 seconds (retry 1 of 4)


[single] turn 2 (openai/gpt-oss-120b, 1,507 tokens)
  -> calculate_volatility(ticker='AAPL')
     why: Need to know if current 30‑day volatility is unusually high or low compared to past year to assess volatility risk
[single] tools
  ok calculate_volatility: {"ticker": "AAPL", "window": 30, "as_of": "2026-10-07", "current_pct": 21.13, "min_1y_pct": 10.29, "median_1y_pct": 23.08, "max_1y_pct": 41.15, "percentile_1y":...


...

[single] tools
  ok llm_sentiment: {"score": 0.14, "label": "neutral", "scored": 5, "failed": 0, "counts": {"positive": 1, "negative": 0, "neutral": 4}, "strongest": [{"headline": "Apple (AAPL) E...


INFO:openai._base_client:Retrying request in 10.000000 seconds (retry 2 of 4)


[single] turn 6 (openai/gpt-oss-120b, 2,844 tokens)
  -> web_search(query='Apple risk factors next quarter 2026 macro, supply chain, regulatory')
     why: Identify analyst commentary on near‑term risks for AAPL to list top three
[single] tools
  ok web_search: {"query": "Apple risk factors next quarter 2026 macro, supply chain, regulatory", "backend": "text", "hits": [{"title": "Apple PESTEL Analysis 2026 - The Strate...


INFO:openai._base_client:Retrying request in 7.000000 seconds (retry 1 of 4)


[single] report written by llm: 3 risks, hedge protective_put, 0 warnings

cycle into turn 2
  observed get_price_data -> ok: {"tool": "get_price_data", "status": "ok", "data": {"ticker": "AAPL", "period": "1mo", "as_of": "2026-10-07", "close": 336.91, "period_return_pct": 6.54, "perio
  chose    calculate_volatility(ticker='AAPL', window=30)
           why: Need to know if current 30‑day volatility is unusually high or low compared to past year to assess volatility risk
cycle into turn 3
  observed calculate_volatility -> ok: {"tool": "calculate_volatility", "status": "ok", "data": {"ticker": "AAPL", "window": 30, "as_of": "2026-10-07", "current_pct": 21.13, "min_1y_pct": 10.29, "med
  chose    get_news(ticker='AAPL', n=10)
           why: Gather recent headlines to gauge sentiment and identify any emerging risks
cycle into turn 4
  observed get_news -> error: {"tool": "get_news", "status": "error", "error": "injected failure: get_news is in failing_tools (a demo switch that is off by

**Same code, different order.** Both runs used the same agent class, prompt and
tools. Only what the tools returned differed. A hard-coded sequence would produce
the same column twice.

In [13]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
rows = max(len(main_order), len(failing_order))
pad = lambda seq: seq + [""] * (rows - len(seq))
order = pd.DataFrame({"3A.2 main run": pad(main_order), "get_news failing": pad(failing_order)}, index=range(1, rows + 1))
order.index.name = "call"
print("failing run's statuses:", [obs["status"] for obs in failing_run.observations])
print("report data gaps:", failing_run.report.data_gaps)
order

failing run's statuses: ['ok', 'ok', 'error', 'ok', 'ok', 'ok']
report data gaps: ['get_news returned no usable result (error: injected failure: get_news is in failing_tools (a demo switch that is off by default))']


,3A.2 main run,get_news failing
call,,
1,get_price_data,get_price_data
2,calculate_volatility,calculate_volatility
3,get_news,get_news
4,llm_sentiment,web_search
5,web_search,llm_sentiment
6,,web_search


In [14]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
bad_agent = ToolSession(today=TODAY, subject=BAD_TICKER)
RUN_LABELS[bad_agent.run_id] = "3A.5 bad ticker, agent"
bad_run = ResearchAgent(bad_agent).research(BAD_TICKER, on_update=print_update)
print("\ncalls:", [(obs["tool"], obs["status"]) for obs in bad_run.observations])
print("report generated_by:", bad_run.report.generated_by)
print("risks:", [risk.title for risk in bad_run.report.top_risks])
print("hedge:", bad_run.report.hedge.strategy, "| levels:", "none (no price data)" if bad_run.report.hedge.levels is None else "computed")
print("data gaps:")
for gap in bad_run.report.data_gaps:
    print(" -", gap)

INFO:openai._base_client:Retrying request in 19.000000 seconds (retry 1 of 4)


[single] turn 1 (openai/gpt-oss-120b, 1,033 tokens)
  -> web_search(query='NOTATICKERZZ stock')
     why: Check if NOTATICKERZZ is a real ticker or find related company info


INFO:openai._base_client:Retrying request in 9.000000 seconds (retry 1 of 4)


[single] tools
  ok web_search: {"query": "NOTATICKERZZ stock", "backend": "text", "hits": [{"title": "FiscalNote Holdings, Inc. - NOTE - Stock Price Today - Zacks", "snippet": "View FiscalNot...
[single] turn 2 (openai/gpt-oss-120b, 1,532 tokens)
  says: I wasn’t able to locate any market data, news or analyst coverage for the ticker **NOTATICKERZZ** – web searches return no relevant results, and the usual price‑history and sentiment APIs return empty responses for this symbol. Because there is no observable price trend, volatility, recent headlines, or analyst commentary, I cannot identify concrete risks or construct a data‑driven hedge for a stock that effectively does not exist in the public markets.


[single] report written by template: 3 risks, hedge trim_and_stop, 7 warnings

calls: [('web_search', 'ok')]
report generated_by: template
risks: ['Trend reversal', 'Volatility', 'News sentiment']
hedge: trim_and_stop | levels: none (no price data)
data gaps:
 - get_price_data was not called
 - calculate_volatility was not called
 - get_news was not called
 - llm_sentiment was not called


## 3B.1 Two agents with a critique loop

A fixed outer graph coordinates two agents, each its own tool loop with its own
tools bound:

`cache_lookup → news_intake → Agent A → Agent B → Agent A answers → final_report → cache_save`

- **news_intake** (code, not an agent) fetches headlines for A, because A has no
  `get_news`.
- **Agent A, the data analyst** (`get_price_data`, `calculate_volatility`,
  `llm_sentiment`), produces a `DataBrief`. Code copies the numbers from A's tool
  results, and A's LLM writes 3–5 key observations.
- **Agent B, the research writer** (`web_search`, `get_news`), reads the brief,
  searches for commentary, and sends **one** `ClarificationRequest` back to A.
- **Agent A answers** with a `ClarificationResponse`, which holds the requested
  data as a typed snapshot. If A doesn't fetch it, the pipeline calls the tool
  for A and labels the answer `fulfilled_by: pipeline`.
- **final_report**: B writes the report from the brief, its own research and the
  answer, and says what the clarification changed.

The critique is a fixed edge, so it runs exactly once on every uncached run. No
step needs a person.

In [15]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.pipeline import ResearchPipeline

pipeline_session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[pipeline_session.run_id] = "3B pipeline"
pipeline_started = time.perf_counter()
first = ResearchPipeline(pipeline_session).run(TICKER, on_update=print_update)
print(f"\nfinished in {time.perf_counter() - pipeline_started:.0f} s | cached: {first.cached} | warnings: {first.warnings or 'none'}")

INFO:task3_agentic.cache:cache miss for /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json: no file
INFO:task1_financial.data:retrying AAPL yahoo news in 1s (attempt 2 of 3): HTTPError: HTTP Error 429: Too Many Requests


[pipeline] cache miss: no valid /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json; running both agents


INFO:task1_financial.data:retrying AAPL yahoo news in 2s (attempt 3 of 3): HTTPError: HTTP Error 429: Too Many Requests
INFO:task1_financial.news:AAPL yahoo feed: 0 items, 0 recent, 0 new
INFO:task1_financial.news:AAPL google feed: 100 items, 53 recent, 52 new


[pipeline] news intake for Agent A (ok): 10 headlines
  - Morgan Stanley Is Bullish on Apple—But With a Catch
  - Apple stock heads toward November 2 earnings with 9% guidance
  - Tim Cook Sells $63.8 Million in Apple Stock After Restricted Shares Vest
  - Apple’s New CEO Just Sold $8.5 Million of Stock
  - Apple (AAPL) Shares Enter 24 7 Token Trading Plan In 63 Stock Filing
  - 21,189 Apple Inc. $AAPL Shares Purchased by Munich Reinsurance Co Stock Corp in Munich
  - Nvidia Is About 4% From Becoming the First $6 Trillion Company. History Says the Milestone Itself Tells You Little.
  - 3 Ways New Investors Get Apple Stock Wrong
  - What Would It Take For Apple Stock To Move Higher?
  - UBS reiterates Apple stock neutral on slowing App Store growth


INFO:task1_financial.data:AAPL: 501 daily bars from 2024-10-08 to 2026-10-07


...

[A] turn 1 (openai/gpt-oss-120b, 1,790 tokens)
  -> get_price_data(period='6mo', ticker='AAPL')
     why: The writer needs a 6‑month series to assess medium‑term momentum and trend, extending beyond the 1‑month snapshot already provided.
[A] tools
  ok get_price_data: {"ticker": "AAPL", "period": "6mo", "as_of": "2026-10-07", "close": 337.2, "period_return_pct": 33.26, "period_high": 345.34, "period_low": 245.26, "high_52w": ...


INFO:openai._base_client:Retrying request in 22.000000 seconds (retry 1 of 4)


[A] turn 2 (openai/gpt-oss-120b, 2,419 tokens)
  says: Over the past six months AAPL has risen 33.3 % to $337.2, staying well above both its 50‑day SMA ($322.3) and 200‑day SMA ($289.7) – +4.6 % and +16.4 % respectively – and trading near the top of its 52‑week range ($345.3 high). Technicals show a neutral‑to‑slightly‑bullish picture: RSI = 58, MACD is still below its signal (‑0.79 histogram), and the price sits at 69 % of the Bollinger Band range (just above the lower band). Volatility remains modest at 30‑day HV ≈ 21 % (32.9 th percentile of the past year), and headline sentiment is essentially neutral (score ≈ ‑0.01).
[A -> B] clarification answer (fulfilled by agent)
  answer: Over the past six months AAPL has risen 33.3 % to $337.2, staying well above both its 50‑day SMA ($322.3) and 200‑day SMA ($289.7) – +4.6 % and +16.4 % respectively – and trading near the top of its 52‑week range ($345.3 high). Technicals show a neutral‑to‑slightly‑bullish picture: RSI = 58, MACD is still be

INFO:openai._base_client:Retrying request in 1.000000 seconds (retry 2 of 4)
INFO:openai._base_client:Retrying request in 10.000000 seconds (retry 3 of 4)
INFO:task3_agentic.cache:saved the research brief to /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json


[B] report written by llm: 3 risks, hedge protective_put, 0 warnings
  clarification used: Added the 6‑month price return of 33.26% (from the clarification’s get_price_data) which reinforced the bullish trend assessment and supported the choice of a protective put given the strong upside cushion.
[pipeline] saved the research brief to /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json

finished in 228 s | cached: False | warnings: none


In [16]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print(f"generated_by: {first.report.generated_by} | tools called (both agents): {first.report.tools_called}")
display(Markdown(first.report.to_markdown()))

generated_by: llm | tools called (both agents): ['get_price_data', 'calculate_volatility', 'llm_sentiment', 'web_search', 'get_news']


# AAPL research report (2026-10-07)

## Financial Health Summary

AAPL closed at $337.2, which is 4.64% above its 50‑day SMA and 16.38% above its 200‑day SMA, indicating a strong bullish trend. Momentum indicators show RSI at 58.2 and a MACD histogram of –0.79, suggesting neutral‑to‑slightly‑bullish momentum. 30‑day historical volatility is 21.16% and sits at the 32.94th percentile of the past year, meaning the stock is calmer than usual. News sentiment is neutral with a score of –0.007, balancing positive earnings guidance (sentiment score 0.85) against insider‑selling headlines.

## Top Three Risks

1. **Insider Selling Pressure**: Large insider sales could depress the share price, potentially pulling it toward the lower end of the 90‑day range.
   - Tim Cook Sells $63.8 Million in Apple Stock After Restricted Shares Vest _(source: get_news)_
   - Apple’s New CEO Just Sold $8.5 Million of Stock _(source: get_news)_
   - Volatility percentile 32.94 (calculate_volatility) _(source: calculate_volatility)_
2. **Earnings Guidance Uncertainty**: The upcoming earnings guidance of 9% growth may be missed, creating downside risk around the November 2 earnings date.
   - Apple stock heads toward November 2 earnings with 9% guidance _(source: get_news)_
   - Apple stock heads toward November 2 earnings with 9% guidance (sentiment score 0.85) _(source: llm_sentiment)_
   - Morgan Stanley Is Bullish on Apple—But With a Catch _(source: web_search)_
3. **Limited Upside Near 52‑Week High**: Trading near the 52‑week high leaves little room for further gains, capping upside and making any pullback more impactful.
   - high_52w 345.34 _(source: get_price_data)_
   - close_vs_sma_200_pct 16.38% _(source: get_price_data)_
   - 30‑day HV ≈ 21.16% (calculate_volatility) _(source: calculate_volatility)_

## Hedge Strategy Recommendation

**Protective Put**

- buy put at 301.53 (one_sd_low)

With the 30‑day volatility at the 32.94th percentile, option premiums are relatively cheap, making a protective put an efficient hedge. Buying a put at the one_sd_low level (≈ $301.5) caps downside from insider selling, a potential earnings miss, and a pullback from the 52‑week high. The put provides full protection down to the bottom of the expected 90‑day range while preserving upside above the current price.

Expected 63-trading-day move: ±35.67 (10.6%) from 337.20, using 21.2% volatility (calculate_volatility, 30-day).

_Option premiums are not priced: no tool returns option chains. Levels are indicative strikes from the expected move and price levels; price the legs from a live chain before acting._

## What the clarification changed

Added the 6‑month price return of 33.26% (from the clarification’s get_price_data) which reinforced the bullish trend assessment and supported the choice of a protective put given the strong upside cushion.

## 3B.2 The handoffs, as typed objects

Every message between the agents is a Pydantic model, validated on both sides
and written to the trace as JSON (`handoff` and `critique` events). Below are the
three objects from the run above.

In [17]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.schemas import ClarificationRequest, ClarificationResponse, DataBrief

for obj in (first.brief, first.request, first.response):
    assert isinstance(obj, (DataBrief, ClarificationRequest, ClarificationResponse))
print("A -> B  DataBrief:")
print(first.brief.model_dump_json(indent=2))

A -> B  DataBrief:
{
  "ticker": "AAPL",
  "as_of": "2026-10-07",
  "headlines_given": 10,
  "price": {
    "ticker": "AAPL",
    "period": "1mo",
    "as_of": "2026-10-07",
    "close": 337.2,
    "period_return_pct": 6.63,
    "close_vs_sma_50_pct": 4.64,
    "close_vs_sma_200_pct": 16.38,
    "high_52w": 345.34,
    "low_52w": 242.76,
    "sma_200": 289.74,
    "rsi_14": 58.2,
    "macd": 3.32,
    "macd_signal": 4.11,
    "macd_hist": -0.79,
    "bb_lower": 327.23,
    "bb_pct_b": 0.69,
    "hv_30_pct": 21.16
  },
  "volatility": {
    "ticker": "AAPL",
    "window": 30,
    "as_of": "2026-10-07",
    "current_pct": 21.157053219863993,
    "min_1y_pct": 10.2865514990347,
    "median_1y_pct": 23.08478035980696,
    "max_1y_pct": 41.150307688507546,
    "percentile_1y": 32.93650793650794,
    "observations": 252
  },
  "sentiment": {
    "score": -0.007,
    "label": "neutral",
    "scored": 10,
    "counts": {
      "positive": 3,
      "negative": 3,
      "neutral": 4
    },
    "

In [18]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
print("B -> A  ClarificationRequest:")
print(first.request.model_dump_json(indent=2))
print("\nA -> B  ClarificationResponse (the request it answers is shown above):")
print(first.response.model_dump_json(indent=2, exclude={"request"}))
print("\nIncorporated: what B's final report says the clarification changed:")
print(first.report.clarification_used)

B -> A  ClarificationRequest:
{
  "question": "Can you provide 6‑month price data (including returns, SMA, RSI, MACD, Bollinger Bands, etc.) for AAPL?",
  "needs": "price_data",
  "period": "6mo",
  "window": null,
  "headlines": null,
  "why": "The brief only includes a 1‑month snapshot, which limits our ability to evaluate medium‑term momentum and trend strength. A 6‑month price series will let us confirm whether the bullish bias is sustained, assess the relevance of the 200‑day SMA, and better gauge potential upside/downside within the 90‑day horizon."
}

A -> B  ClarificationResponse (the request it answers is shown above):
{
  "answer": "Over the past six months AAPL has risen 33.3 % to $337.2, staying well above both its 50‑day SMA ($322.3) and 200‑day SMA ($289.7) – +4.6 % and +16.4 % respectively – and trading near the top of its 52‑week range ($345.3 high). Technicals show a neutral‑to‑slightly‑bullish picture: RSI = 58, MACD is still below its signal (‑0.79 histogram), and th

## 3B.3 Tool restriction

Restriction is enforced by **binding**: each agent's model is given only its own
tools, so it cannot even name another. If a model asks for a tool it wasn't
given, the tools node refuses the call and traces it as `refused`
(`tests/test_task3_pipeline.py::test_writer_cannot_call_price_tools`). The
crosstab counts the real run's traced calls by agent. `pipeline` is the news
intake step, which is code.

In [19]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
from task3_agentic.handoff import ANALYST_TOOLS
from task3_agentic.pipeline import WRITER_TOOLS

print("Agent A is bound to:", ANALYST_TOOLS)
print("Agent B is bound to:", WRITER_TOOLS)
calls = pd.DataFrame([e for e in read_trace(TRACE, run_id=pipeline_session.run_id) if e["event"] == "tool_call"])
print("refused calls:", int((calls["status"] == "refused").sum()))
pd.crosstab(calls["agent"], calls["tool"])

Agent A is bound to: ('get_price_data', 'calculate_volatility', 'llm_sentiment')
Agent B is bound to: ('web_search', 'get_news')
refused calls: 0


tool,calculate_volatility,get_news,get_price_data,llm_sentiment,web_search
agent,,,,,
A,1,0,2,1,0
B,0,1,0,0,2
pipeline,0,1,0,0,0


## 3C.2 Persistent cache

After an uncached run, the pipeline saves the report, the brief and the
clarification exchange to `task3_agentic/cache/{TICKER}_{date}.json` (the
market's date in New York). Below, a **new** pipeline with a new tool session,
which stands in for a later run, finds the file, validates it against the
schema, and returns it without calling a single tool. A corrupt or old-schema
file counts as a miss. A template-written report is never cached.

In [20]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
before = tool_call_count()
later_session = ToolSession(today=TODAY, subject=TICKER)
RUN_LABELS[later_session.run_id] = "3C.2 cache hit"
second = ResearchPipeline(later_session).run(TICKER, on_update=print_update)
new_lines = tool_call_count() - before
assert second.cached and new_lines == 0, (second.cached, new_lines)
assert second.report == first.report
print(f"cache hit: {second.cached} | new tool_call lines: {new_lines} | same report as the first run: {second.report == first.report}")
saved = json.loads(CACHE_FILE.read_text())
print(f"{CACHE_FILE.relative_to(ROOT)}: {CACHE_FILE.stat().st_size:,} bytes, keys {list(saved)}")
print({key: saved[key] for key in ("run_id", "profile", "created_at", "schema_version")})

INFO:task3_agentic.cache:cache hit: /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json (saved 2026-10-07 18:15:55.329456+00:00)


[pipeline] cache hit: /content/CDAZZDEV-MLE-SATHURSAN/task3_agentic/cache/AAPL_2026-10-07.json (saved 2026-10-07T18:15:55.329456+00:00); loading it instead of running the agents
cache hit: True | new tool_call lines: 0 | same report as the first run: True
task3_agentic/cache/AAPL_2026-10-07.json: 11,808 bytes, keys ['schema_version', 'ticker', 'as_of', 'run_id', 'profile', 'created_at', 'report', 'brief', 'request', 'response']
{'run_id': 'bf170e93ef95', 'profile': 'free', 'created_at': '2026-10-07T18:15:55.329456Z', 'schema_version': 1}


## 3C.3 `agent_trace.jsonl`

Every tool call, whether made by code or by an agent, goes through
`ToolSession.call`, which writes one `tool_call` line: tool, args, the agent's
`why`, status, output (the digest, cut to 200 characters, with its full length
in `output_chars`), wall-clock `duration_ms`, and `cache_hit`. Model turns,
reports, handoffs, critiques and cache lookups are their own events. The
committed `task3_agentic/logs/agent_trace.jsonl` is this run's file.

In [21]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
events = pd.DataFrame(read_trace(TRACE))
events["section"] = events["run_id"].map(RUN_LABELS)
print(f"{len(events)} events in {TRACE.relative_to(ROOT)}")
events.pivot_table(index="section", columns="event", values="ts", aggfunc="count", fill_value=0)

65 events in task3_agentic/logs/agent_trace.jsonl


event,agent_turn,cache,critique,handoff,report,tool_call
section,,,,,,
3A.1 direct tools,0,0,0,0,0,5
3A.2 research agent,7,0,0,0,1,5
"3A.5 bad ticker, agent",2,0,0,0,1,1
"3A.5 bad ticker, direct",0,0,0,0,0,5
3A.5 get_news failing,6,0,0,0,1,6
3B pipeline,10,2,2,1,1,8
3C.2 cache hit,0,1,0,0,0,0


In [22]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
tool_calls = events[events["event"] == "tool_call"]
assert {"tool", "args", "output", "duration_ms"} <= set(tool_calls.columns)
print(f"{len(tool_calls)} tool calls; longest stored output: {tool_calls['output'].str.len().max()} characters")
tool_calls[["section", "agent", "tool", "args", "status", "duration_ms", "cache_hit", "output"]].head(12)

30 tool calls; longest stored output: 200 characters


,section,agent,tool,args,status,duration_ms,cache_hit,output
0,3A.1 direct tools,direct,get_price_data,"{'ticker': 'AAPL', 'period': '6mo'}",ok,881.0,False,"{""tool"": ""get_price_data"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""period"": ""6mo"", ""as_of"": ""2026-10-07"", ""close..."
1,3A.1 direct tools,direct,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}",ok,4.7,True,"{""tool"": ""calculate_volatility"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""window"": 30, ""as_of"": ""2026-10-07"", ""cu..."
2,3A.1 direct tools,direct,get_news,"{'ticker': 'AAPL', 'n': 10}",ok,4135.1,False,"{""tool"": ""get_news"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""count"": 10, ""headlines"": [{""title"": ""Morgan Stanley..."
3,3A.1 direct tools,direct,llm_sentiment,"{'headlines': ['Morgan Stanley Is Bullish on Apple—But With a Catch', 'Apple stock heads toward November 2 earnings ...",ok,5035.7,False,"{""tool"": ""llm_sentiment"", ""status"": ""ok"", ""data"": {""score"": -0.007, ""label"": ""neutral"", ""scored"": 10, ""failed"": 0, ""..."
4,3A.1 direct tools,direct,web_search,{'query': 'AAPL stock analyst outlook next quarter'},ok,2216.7,False,"{""tool"": ""web_search"", ""status"": ""ok"", ""data"": {""query"": ""AAPL stock analyst outlook next quarter"", ""backend"": ""text..."
6,3A.2 research agent,single,get_price_data,"{'ticker': 'AAPL', 'period': '1mo'}",ok,333.9,False,"{""tool"": ""get_price_data"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""period"": ""1mo"", ""as_of"": ""2026-10-07"", ""close..."
8,3A.2 research agent,single,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}",ok,4.1,True,"{""tool"": ""calculate_volatility"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""window"": 30, ""as_of"": ""2026-10-07"", ""cu..."
10,3A.2 research agent,single,get_news,"{'ticker': 'AAPL', 'n': 10}",ok,3287.0,False,"{""tool"": ""get_news"", ""status"": ""ok"", ""data"": {""ticker"": ""AAPL"", ""count"": 10, ""headlines"": [{""title"": ""Morgan Stanley..."
12,3A.2 research agent,single,llm_sentiment,"{'headlines': ['Morgan Stanley Is Bullish on Apple—But With a Catch', 'Apple stock heads toward November 2 earnings ...",ok,2712.6,False,"{""tool"": ""llm_sentiment"", ""status"": ""ok"", ""data"": {""score"": -0.007, ""label"": ""neutral"", ""scored"": 10, ""failed"": 0, ""..."
14,3A.2 research agent,single,web_search,{'query': 'Apple AAPL risk factors next 90 days insider selling token trading plan regulatory'},ok,996.0,False,"{""tool"": ""web_search"", ""status"": ""ok"", ""data"": {""query"": ""Apple AAPL risk factors next 90 days insider selling token..."


In [23]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
tool_calls.groupby("tool").agg(
    calls=("ts", "size"),
    ok=("status", lambda s: int((s == "ok").sum())),
    mean_ms=("duration_ms", "mean"),
    max_ms=("duration_ms", "max"),
    cache_hits=("cache_hit", "sum"),
).round(0)

,calls,ok,mean_ms,max_ms,cache_hits
tool,,,,,
calculate_volatility,5,4,619.0,3079.0,4
get_news,6,4,3553.0,6558.0,0
get_price_data,6,5,1053.0,4533.0,1
llm_sentiment,5,4,6199.0,17579.0,0
web_search,8,8,1605.0,2217.0,0


## Run summary

In [24]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
calls_log = pd.read_json(CALL_LOG, lines=True) if CALL_LOG.exists() else pd.DataFrame()
llm_tokens = int(calls_log[["prompt_tokens", "completion_tokens"]].sum().sum()) if len(calls_log) else 0
agent_tokens = int(events.loc[events["event"] == "agent_turn", "tokens"].fillna(0).sum())
failed_at = next((i for i, obs in enumerate(failing_run.observations) if obs["tool"] == "get_news" and obs["status"] != "ok"), None)
switched = failing_order[failed_at + 1:] if failed_at is not None else []
print(f"Model:            {profile.name} ({profile.primary.model})")
print(f"3A report:        {report.generated_by}; risks: {[risk.title for risk in report.top_risks]}")
print(f"3A hedge:         {report.hedge.strategy}")
print(f"3A follow-up:     {follow_up.tool_calls} tool calls")
print(f"get_news failing: after the failure the agent called {switched}")
print(f"Bad ticker:       report by {bad_run.report.generated_by}, {len(bad_run.report.data_gaps)} data gaps, no exception")
print(f"3B request:       needs {first.request.needs}: {first.request.question}")
print(f"3B answer:        fulfilled by {first.response.fulfilled_by}; report by {first.report.generated_by}")
print(f"3C cache:         second run cached={second.cached}")
print(f"Trace:            { {event: int(count) for event, count in events['event'].value_counts().items()} }")
print(f"Tokens:           {agent_tokens:,} in agent turns + {llm_tokens:,} in structured calls ({len(calls_log)} logged)")
print(f"Wall time:        {time.perf_counter() - RUN_STARTED:.0f} s")

Model:            free (openai/gpt-oss-120b)
3A report:        llm; risks: ['Insider Selling Pressure', 'Token‑Trading Plan Regulatory Uncertainty', 'App Store Growth Slowdown Concerns']
3A hedge:         protective_put
3A follow-up:     0 tool calls
get_news failing: after the failure the agent called ['web_search', 'llm_sentiment', 'web_search']
Bad ticker:       report by template, 4 data gaps, no exception
3B request:       needs price_data: Can you provide 6‑month price data (including returns, SMA, RSI, MACD, Bollinger Bands, etc.) for AAPL?
3B answer:        fulfilled by agent; report by llm
3C cache:         second run cached=True
Trace:            {'tool_call': 30, 'agent_turn': 25, 'report': 4, 'cache': 3, 'critique': 2, 'handoff': 1}
Tokens:           50,435 in agent turns + 27,058 in structured calls (9 logged)
Wall time:        533 s


In [25]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Start PR 4: the Task 3 notebook, laid out as settled in the grilling rounds', Date: 2026-10-07
# Colab only: download the trace, the call log and the cache file, which are committed beside this notebook.
import zipfile

if IN_COLAB:
    from google.colab import files

    bundle = Path("/content/task3_run_outputs.zip")
    with zipfile.ZipFile(bundle, "w") as archive:
        for path in (TRACE, CALL_LOG, CACHE_FILE):
            if path.exists():
                archive.write(path, path.relative_to(ROOT))
        print("bundled:", archive.namelist())
    files.download(str(bundle))
else:
    print("Local run: the files are already in place:", [str(p.relative_to(ROOT)) for p in (TRACE, CALL_LOG, CACHE_FILE) if p.exists()])

bundled: ['task3_agentic/logs/agent_trace.jsonl', 'task3_agentic/logs/llm_calls.jsonl', 'task3_agentic/cache/AAPL_2026-10-07.json']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>